# neuralCAD-Edit 阅读入口
先阅读 README.md 与 PAPER_AND_CODE.md。此 notebook 只读取发布数据；不调用模型 API。请从本目录启动 JupyterLab。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
assert (ROOT / 'study.py').exists(), '请在 research/neuralcad_edit 启动 JupyterLab'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '.local/upstream'))
import study
db = study.open_database()
requests = list(db.requests.find({'request_type':'edit'}))
print('requests:', len(requests))
print('modalities:', sorted({r.get('modality') for r in requests}))

In [ ]:
from IPython.display import display, Image, Video
request, step = study.select_request(db)
display({k: request.get(k) for k in ['_id','modality','difficulty','text','prompt','transcript']})
print('Input STEP:', step)
for path in db.get_brep_images(request['brep_start'], views=['toprightiso']):
    display(Image(filename=str(study.WORKSPACE / path)))

选择其他任务：查看 requests 中的 _id，然后调用 study.select_request(db, request_id)。可以分别查看原始输入、请求视频、请求者 GT 和另一位专家的输出。研究时先观察请求与输入，再查看 GT，避免把目标答案当作模型输入。

In [ ]:
gt = db.edits.find_one({'request':request['_id'], 'user':request['user']})
other = next((e for e in db.edits.find({'request':request['_id']}) if e['user'] != request['user'] and db.users.find_one({'_id':e['user']}).get('is_human', True)), None)
for label, edit in [('requestor GT',gt), ('other expert',other)]:
    print(label)
    if edit:
        for path in db.get_brep_images(edit['brep_end'], views=['toprightiso']):
            display(Image(filename=str(study.WORKSPACE / path)))

In [ ]:
import pandas as pd
result = ROOT / '.local/published-leaderboard.csv'
if result.exists():
    display(pd.read_csv(result))
db.close_connection()